# tRNA41 Contact Prediction Benchmark

Systematically compare methods and models for recovering tRNA secondary structure
from genomic foundation model embeddings.

**Methods:**
- Embedding perturbation map (cosine, L2, custom diffs)
- Log-odds dependency map (MLM models only)
- Epistasis metrics (from precomputed DBs)

**Goal:** Find the best (method, model, metric) combination for contact prediction.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from seqmat import SeqMat

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_logodds_dependency_map,
    compute_embedding_perturbation_map,
    compute_nucleotide_dependency_map,
)

In [ ]:
# --- tRNA41 (tRNA-Arg-TCT-4-1) coordinates (minus strand) ---
CHROM = 'chr1'
START = 159141611
END = 159141684
PAD = 256

# Dependency maps must run on the RC sequence (tRNA is on minus strand)
s_padded = SeqMat.from_fasta('hg38', CHROM, START - PAD, END + PAD)
s_padded.reverse_complement()
seq_rc_padded = s_padded.seq
tRNA_positions = list(range(PAD, PAD + END - START + 1))
print(f"tRNA region: {END - START + 1} bp, padded RC sequence: {len(seq_rc_padded)} bp")

## 2. Ground truth: tRNA secondary structure

In [ ]:
def dotbracket_to_contact_map(ss):
    """Convert dot-bracket notation to a symmetric binary contact matrix."""
    opens = "([{<ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    closes = ")]}>abcdefghijklmnopqrstuvwxyz"
    N = len(ss)
    M = np.zeros((N, N), dtype=int)
    stack = {ch: [] for ch in opens}
    for i, ch in enumerate(ss):
        if ch in opens:
            stack[ch].append(i)
        elif ch in closes:
            j = stack[opens[closes.index(ch)]].pop()
            M[i, j] = M[j, i] = 1
    return M


ss_raw = '.>>>>>>>..>>>>..........<<<<.>>>>>.........................<<<<<................>>>>>.......<<<<<<<<<<<<.'
ss_raw = ss_raw.replace('>', '(').replace('<', ')')
refseq_raw = '.GTCTCTGTGGCGCAATGGAcgA.GCGCGCTGGACTTCTA..................ATCCAGAG...........GtTCCGGGTTCGAGTCCCGGCAGAGATG'
ss = ''.join(c for c, r in zip(ss_raw, refseq_raw) if r != '.')
M_true = dotbracket_to_contact_map(ss)

plt.figure(figsize=(6, 6))
plt.imshow(M_true, cmap='viridis', origin='upper')
plt.title("tRNA41 ground truth contact map")
plt.xlabel("Position")
plt.ylabel("Position")
plt.colorbar()
plt.show()
print(f"Structure: {len(ss)} nt, {M_true.sum() // 2} base pairs")

In [ ]:
def upper_tri_flatten(M):
    i, j = np.triu_indices_from(M, k=1)
    return M[i, j]


def zero_diagonal(M):
    """Zero out the diagonal of a matrix."""
    M = M.copy()
    np.fill_diagonal(M, 0)
    return M


def score_map(pred_matrix, M_true, flip=False):
    """Score a predicted contact map against ground truth.
    
    Zeros out the diagonal of both maps before comparing.
    Returns (pearson_r, spearman_r, pearson_p).
    
    flip=False by default because we run on RC sequence directly.
    """
    pred = pred_matrix.copy()
    if flip:
        pred = pred[::-1, ::-1]
    pred[np.isnan(pred)] = 0
    pred = zero_diagonal(pred)
    truth = zero_diagonal(M_true)
    true_flat = upper_tri_flatten(truth)
    pred_flat = upper_tri_flatten(pred)
    r_pear, p_pear = pearsonr(true_flat, pred_flat)
    r_spear, _ = spearmanr(true_flat, pred_flat)
    return r_pear, r_spear, p_pear


# Collect all results here
results = []

## 3. Define models to benchmark

Uncomment models as needed. Each entry is `(name, wrapper_class, init_kwargs, context, compute_kwargs)`.

`compute_kwargs` are passed through to `compute_*` functions (e.g. `species_proxy` for SpeciesLM).

In [ ]:
from genebeddings.wrappers import (
    RiNALMoWrapper,
    SpeciesLMWrapper,
    # NTWrapper,
    # MutBERTWrapper,
    # HyenaDNAWrapper,
    # CaduceusWrapper,
    # ConvNovaWrapper,
)

# (name, wrapper_class, init_kwargs, context, compute_kwargs)
# SpeciesLM fungi (k=1) — best multispecies model for tRNA structure (Tomaz Da Silva et al. 2025)
# SpeciesLM metazoa (k=6) — trained on metazoan genomes, uses homo_sapiens proxy
MODELS = [
    ('RiNALMo',              RiNALMoWrapper,   {},                          511, {}),
    ('SpeciesLM (fungi)',     SpeciesLMWrapper,  {'preset': 'fungi'},        1024, {}),
    ('SpeciesLM (metazoa)',   SpeciesLMWrapper,  {'preset': 'metazoa'},      8192, {}),
    # ('NT-500m',     NTWrapper,         {'model_name': 'v2-500m-multi'}, 3000, {}),
    # ('NT-250m',     NTWrapper,         {'model_name': 'v2-250m-multi'}, 3000, {}),
    # ('NT-2.5b',     NTWrapper,         {'model_name': '2.5b-multi'},    3000, {}),
    # ('MutBERT',     MutBERTWrapper,    {},                       256, {}),
    # ('HyenaDNA',    HyenaDNAWrapper,   {},                       60_000, {}),
    # ('Caduceus',    CaduceusWrapper,   {},                       60_000, {}),
    # ('ConvNova',    ConvNovaWrapper,   {},                       1000, {}),
]

## 4. Embedding perturbation maps

Try every (model, diff, aggregation) combination.

In [ ]:
import torch


def mahal_distance(cov_inv):
    """Return a Mahalanobis distance function using the given inverse covariance."""
    cov_inv_t = torch.tensor(cov_inv, dtype=torch.float32)
    def _mahal(x, y, eps=1e-20):
        d = (x - y).unsqueeze(0).float()
        return torch.sqrt(d @ cov_inv_t.to(d.device) @ d.T + eps).item()
    return _mahal


DIFFS = [
    ('cosine', 'cosine'),
    ('L2', 'l2'),
]

AGGREGATIONS = ['max', 'mean']

for model_name, model_cls, model_kwargs, context, compute_kwargs in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    model = model_cls(**model_kwargs)

    for diff_name, diff_fn in DIFFS:
        for agg in AGGREGATIONS:
            label = f"{model_name} | perturbation | {diff_name} | {agg}"
            print(f"  Running: {label}")
            result = compute_embedding_perturbation_map(
                model, seq_rc_padded,
                positions=tRNA_positions,
                diff=diff_fn,
                aggregation=agg,
                show_progress=True,
                **compute_kwargs,
            )
            r_pear, r_spear, p = score_map(result.matrix, M_true)
            print(f"    Pearson r={r_pear:.4f}, Spearman r={r_spear:.4f}, p={p:.2e}")
            results.append({
                'model': model_name,
                'method': 'perturbation',
                'diff': diff_name,
                'aggregation': agg,
                'pearson_r': r_pear,
                'spearman_r': r_spear,
                'pearson_p': p,
            })

    del model

## 5. Log-odds dependency maps (MLM models only)

In [ ]:
for model_name, model_cls, model_kwargs, context, compute_kwargs in MODELS:
    model = model_cls(**model_kwargs)

    # Skip models without MLM head
    if not model.supports_capability('predict_nucleotides'):
        print(f"Skipping {model_name} (no MLM head)")
        del model
        continue

    label = f"{model_name} | log-odds"
    print(f"Running: {label}")
    result = compute_logodds_dependency_map(
        model, seq_rc_padded,
        positions=tRNA_positions,
        show_progress=True,
        **compute_kwargs,
    )
    r_pear, r_spear, p = score_map(result.matrix, M_true)
    print(f"  Pearson r={r_pear:.4f}, Spearman r={r_spear:.4f}, p={p:.2e}")
    results.append({
        'model': model_name,
        'method': 'log-odds',
        'diff': 'KL',
        'aggregation': 'sum',
        'pearson_r': r_pear,
        'spearman_r': r_spear,
        'pearson_p': p,
    })

    del model

## 5b. Nucleotide dependency maps (Tomaz Da Silva et al. 2025)

The paper's method: feed full (unmasked) mutated sequences, score with logit differences.
Much faster than masked log-odds (223 forward passes vs ~130k for SpeciesLM k=1).

In [ ]:
for model_name, model_cls, model_kwargs, context, compute_kwargs in MODELS:
    model = model_cls(**model_kwargs)

    # Skip models without MLM head
    if not model.supports_capability('predict_nucleotides'):
        print(f"Skipping {model_name} (no MLM head)")
        del model
        continue

    label = f"{model_name} | nuc-dependency"
    print(f"Running: {label}")
    result = compute_nucleotide_dependency_map(
        model, seq_rc_padded,
        positions=tRNA_positions,
        show_progress=True,
        **compute_kwargs,
    )
    r_pear, r_spear, p = score_map(result.matrix, M_true)
    print(f"  Pearson r={r_pear:.4f}, Spearman r={r_spear:.4f}, p={p:.2e}")
    results.append({
        'model': model_name,
        'method': 'nuc-dependency',
        'diff': 'logit',
        'aggregation': 'max',
        'pearson_r': r_pear,
        'spearman_r': r_spear,
        'pearson_p': p,
    })

    del model

## 6. Epistasis metrics (from precomputed DBs)

Add entries for any model that has a precomputed embedding DB.

In [ ]:
from itertools import combinations, product

GENE = 'tRNA41'
CHROM_NUM = '1'
STRAND = 'N'
MAX_DISTANCE = 100
BASES = 'ATGC'


def get_all_epistasis_ids(seq, indices, gene, chrom, zyg="N", max_distance=100):
    items = []
    for i, j in combinations(range(len(seq)), 2):
        if j - i > max_distance:
            continue
        pos1, pos2 = int(indices[i]), int(indices[j])
        ref1, ref2 = seq[i], seq[j]
        if ref1 not in BASES or ref2 not in BASES:
            continue
        alts1 = [b for b in BASES if b != ref1]
        alts2 = [b for b in BASES if b != ref2]
        for alt1, alt2 in product(alts1, alts2):
            id1 = f"{gene}:{chrom}:{pos1}:{ref1}:{alt1}:{zyg}"
            id2 = f"{gene}:{chrom}:{pos2}:{ref2}:{alt2}:{zyg}"
            items.append({'epistasis_id': f"{id1}|{id2}", 'pos1': pos1, 'pos2': pos2})
    return pd.DataFrame(items)


s = SeqMat.from_fasta('hg38', CHROM, START, END)
df_epi = get_all_epistasis_ids(s.seq, s.index, GENE, CHROM_NUM, STRAND, MAX_DISTANCE)
print(f"{len(df_epi)} epistasis pairs")

In [ ]:
# Configure DBs for epistasis benchmarking
# Each entry: (model_name, wrapper_cls, wrapper_kwargs, db_path, context)
EPI_DBS = [
    ('RiNALMo', RiNALMoWrapper, {}, 'embeddings/rinalmo.db', 511),
    # ('SpeciesLM', SpeciesLMWrapper, {}, 'embeddings/specieslm.db', 600),
    # ('NT-500m', NTWrapper, {'model_name': 'v2-500m-multi'}, 'embeddings/nt500_multi.db', 3000),
]

EPI_METRICS = ['epi_R_raw', 'epi_R_singles', 'cos_v1_v2', 'magnitude_ratio', 'log_magnitude_ratio']

for model_name, model_cls, model_kwargs, db_path, context in EPI_DBS:
    print(f"\n{'='*60}")
    print(f"Epistasis: {model_name}")
    print(f"{'='*60}")

    model = model_cls(**model_kwargs)
    db = VariantEmbeddingDB(db_path)

    epi_data = add_epistasis_metrics(
        df_epi,
        db,
        model=model,
        id_col="epistasis_id",
        context=context,
        show_progress=True,
        force=False,
        batch_size=8,
    )

    for metric in EPI_METRICS:
        if metric not in epi_data.columns:
            continue
        dep = epi_data.pivot_table(index='pos1', columns='pos2', values=metric, aggfunc='max')
        dep = dep.combine_first(dep.T).fillna(0)
        r_pear, r_spear, p = score_map(dep.values, M_true, flip=True)
        print(f"  {metric}: Pearson r={r_pear:.4f}, Spearman r={r_spear:.4f}")
        results.append({
            'model': model_name,
            'method': 'epistasis',
            'diff': metric,
            'aggregation': 'max',
            'pearson_r': r_pear,
            'spearman_r': r_spear,
            'pearson_p': p,
        })

    del model

## 7. Results

In [ ]:
df_results = pd.DataFrame(results).sort_values('pearson_r', ascending=False).reset_index(drop=True)
df_results['rank'] = range(1, len(df_results) + 1)
df_results

In [ ]:
# Bar chart of Pearson r by method
fig, ax = plt.subplots(figsize=(12, max(4, len(df_results) * 0.4)))
colors = df_results['method'].map({'perturbation': 'C0', 'log-odds': 'C1', 'epistasis': 'C2', 'nuc-dependency': 'C3'})
bars = ax.barh(
    range(len(df_results)),
    df_results['pearson_r'],
    color=colors,
)
ax.set_yticks(range(len(df_results)))
ax.set_yticklabels(
    df_results.apply(lambda r: f"{r['model']} | {r['method']} | {r['diff']} | {r['aggregation']}", axis=1),
    fontsize=9,
)
ax.set_xlabel('Pearson r (vs tRNA secondary structure)')
ax.set_title('tRNA41 Contact Prediction Benchmark')
ax.invert_yaxis()
ax.axvline(0, color='gray', linewidth=0.5)

# Legend
from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color='C0', label='perturbation'), Patch(color='C1', label='log-odds'), Patch(color='C2', label='epistasis'), Patch(color='C3', label='nuc-dependency')],
    loc='lower right',
)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmaps for top 3
print("Top 3 methods:")
for i, row in df_results.head(3).iterrows():
    print(f"  {row['rank']}. {row['model']} | {row['method']} | {row['diff']} | {row['aggregation']}  "
          f"r={row['pearson_r']:.4f}")